# Exp B — Is a threat a zero-day? (Def 13, leave-one-family-out)
Hold each attack family out as 'unseen', measure AUROC of novelty `u`. Compares deployable variants (φ / metadata) vs oracle upper bounds (A/σ from the label) vs raw-feature baselines.

In [ ]:
import sys; sys.path.insert(0, '../src')
import matplotlib.pyplot as plt
from zeroday_verify import data as D
from zeroday_verify.experiments import exp_b_leave_one_family_out, fold_scores_for_family
import pandas as pd
paths = D.discover_csvs('../data/raw')
df = D.load_subsampled(paths, per_attack_family=500, n_benign=2500, seed=17)
threats = D.build_threats(df)
b = exp_b_leave_one_family_out(threats)
print('aggregated AUROC:', {k: round(v,3) for k,v in b['aggregated'].items()})

In [ ]:
rows = pd.DataFrame(b['rows'])
deploy = ['semantic_only','semantic+metadata','raw_1nn','isolation_forest']
rows.pivot(index='held_out', columns='variant', values='auroc')[deploy].round(3)

In [ ]:
from zeroday_verify.metrics import roc_points
fold = fold_scores_for_family(threats, 'PortScan')
for v in ['semantic_only','raw_1nn','composite (oracle A,sigma)']:
    fpr,tpr = roc_points(fold['scores'][v], fold['y']); plt.plot(fpr,tpr,label=v)
plt.plot([0,1],[0,1],':',c='gray'); plt.legend(); plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title("ROC — PortScan held out as zero-day"); plt.show()

**Caveat:** the near-perfect *oracle* variants use `A`/`σ` derived from the family label and are an upper bound, not a deployable result. The trustworthy zero-day signal is `semantic_only` / `semantic+metadata`.